# Fine-tune Qwen3-4B for text-to-SQL — Kaggle (free GPU)Kaggle gives 30 GPU-hours a week, which is enough to run this many times over.**Before you run anything, set three things in the right-hand sidebar:**| Setting | Value ||---|---|| Accelerator | `GPU T4 x2` or `GPU P100` || Internet | **On** (needs phone verification on your Kaggle account) || Add-ons → Secrets | Add `HF_TOKEN` with your Hugging Face **write** token |Using a Secret rather than pasting the token in a cell keeps it out of thenotebook, out of your commit history, and out of any public copy of this kernel.Only one GPU is used — Unsloth's multi-GPU support is a paid feature, and asingle T4 is plenty for a 4B model at 4-bit.

In [ ]:
# --- 1. What hardware did we actually get? -----------------------------------import subprocess, sys, platformprint(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())print("python", platform.python_version())import torchcc = torch.cuda.get_device_capability()print(f"torch {torch.__version__} | capability {cc[0]}.{cc[1]} | bf16 supported: {torch.cuda.is_bf16_supported()}")if not torch.cuda.is_bf16_supported():    print("\n  T4 is Turing, so there is no bf16. train.py selects fp16 automatically.")

## 2. Install UnslothThis is the cell most likely to need adjusting — Kaggle's preinstalled torchmoves around, and Unsloth pins against it. If the import check at the bottomfails, try the fallback in the next cell and restart the session.

In [ ]:
%%capture# Primary install path.!pip install -q --upgrade pip!pip install -q unsloth unsloth_zoo!pip install -q --no-deps trl peft accelerate bitsandbytes!pip install -q datasets huggingface_hub

In [ ]:
# --- Verify the install before spending GPU time on it -----------------------try:    from unsloth import FastLanguageModel, is_bfloat16_supported    import trl, peft, transformers, datasets    print("unsloth imported OK")    print(f"  transformers {transformers.__version__} | trl {trl.__version__} | peft {peft.__version__}")    print(f"  bf16 path: {is_bfloat16_supported()}")except Exception as e:    print("INSTALL FAILED:", type(e).__name__, e)    print("\nFallback — run this, then Run > Restart session, then skip the install cell:")    print("  !pip install -q --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo")

## 3. Get the training codeTwo ways, whichever you prefer. The notebook tries them in order.1. **Clone from GitHub** — set `REPO_URL` below to your pushed repository.2. **Kaggle Dataset** — upload the `training/` folder as a private Dataset and   attach it with *Add Input*. Works without making the repo public.

In [ ]:
# --- 3. Locate the training scripts ------------------------------------------import os, glob, shutil, subprocessfrom pathlib import PathREPO_URL = ""   # e.g. "https://github.com/bharatverse11/text2sql-qwen.git"WORK = Path("/kaggle/working/ft")NEEDED = ["prepare_data.py", "train.py", "evaluate.py", "evalcore.py",          "sqlutil.py", "merge_and_push.py"]def have_all(d: Path) -> bool:    return d.is_dir() and all((d / f).exists() for f in NEEDED)if have_all(WORK):    print(f"Already present at {WORK}")elif REPO_URL:    shutil.rmtree(WORK, ignore_errors=True)    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(WORK.parent / "repo")], check=True)    src = WORK.parent / "repo" / "training"    shutil.copytree(src, WORK, dirs_exist_ok=True)    print(f"Cloned -> {WORK}")else:    # Fall back to an attached Kaggle Dataset containing the training files.    hits = [Path(p).parent for p in glob.glob("/kaggle/input/**/sqlutil.py", recursive=True)]    if not hits:        raise SystemExit(            "No code found. Either set REPO_URL above, or upload the training/ "            "folder as a Kaggle Dataset and attach it with Add Input."        )    WORK.mkdir(parents=True, exist_ok=True)    for f in hits[0].iterdir():        if f.is_file():            shutil.copy(f, WORK / f.name)    print(f"Copied from attached dataset {hits[0]} -> {WORK}")os.chdir(WORK)missing = [f for f in NEEDED if not Path(f).exists()]print("missing:", missing if missing else "none")

In [ ]:
# --- 4. Hugging Face token, from Kaggle Secrets ------------------------------import osfrom kaggle_secrets import UserSecretsClientos.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")HF_REPO = "bharatverse11/qwen3-4b-text2sql"   # where the merged model is publishedfrom huggingface_hub import whoamiprint("authenticated as:", whoami(token=os.environ["HF_TOKEN"])["name"])

## 5. Prepare the dataEvery example is verified to execute against a real in-memory SQLite databasebuilt from its own schema. About 21% of the raw dataset has reference SQL thatdoes not run; those are dropped rather than trained on.

In [ ]:
!python prepare_data.py --train-size 8000 --val-size 200 --test-size 300

## 6. Train`--max-seq 640` is measured, not guessed: the prepared data has a median of 194tokens and a longest example of 581. The usual 2048 default wastes memory andtime on padding that never gets used.Watch the first ~50 steps. The progress bar's `it/s` tells you the real runtimeon whichever GPU Kaggle handed you — multiply out before walking away.

In [ ]:
!python train.py \    --data data \    --out outputs/qwen3-4b-text2sql-lora \    --max-seq 640 \    --batch-size 8 \    --grad-accum 2

## 7. Evaluate: base vs fine-tunedScored by **execution accuracy** — build the database from each test example'sschema, run both the generated and the reference query, compare result sets. Acorrect query written differently still counts.Both models get the identical prompt and greedy decoding.

In [ ]:
!python evaluate.py --adapter outputs/qwen3-4b-text2sql-lora --limit 300

In [ ]:
# --- 8. Merge the adapter into 16-bit weights and publish --------------------!python merge_and_push.py \    --adapter outputs/qwen3-4b-text2sql-lora \    --repo {HF_REPO}

## 9. Collect the results`eval_report.json` is what the website's Benchmark tab renders. Download it fromthe notebook's **Output** panel on the right, then commit it to the repo:```cp eval_report.json web/data/eval_report.json```Then point the Hugging Face Space at the published model and deploy thefrontend — see the repo README, steps 3 and 4.

In [ ]:
import json, shutilfrom pathlib import Pathreport = Path("outputs/eval_report.json")if not report.exists():    raise SystemExit("No eval report - did the evaluate cell finish?")shutil.copy(report, "/kaggle/working/eval_report.json")r = json.loads(report.read_text())b, t = r["metrics"]["base"], r["metrics"]["tuned"]print(f"{'metric':<22}{'base':>10}{'fine-tuned':>13}{'delta':>10}")print("-" * 55)for k in b:    print(f"{k:<22}{b[k]:>9.1%}{t[k]:>12.1%}{t[k]-b[k]:>+9.1%}")c = r["counts"]print(f"\nwins {c['tuned_win']} | both right {c['both_correct']} "      f"| regressions {c['tuned_regression']} | both wrong {c['both_wrong']}")print("\nSaved to /kaggle/working/eval_report.json - download it from the Output panel.")